# 02 – Preprocessing & Data Cleaning

### Purpose of the Notebook
This notebook applies systematic cleaning and standardisation to the pre‑saved datasets (dataset.pkl and dataset_de.pkl).
All decisions are based on the insights from Notebook 01_data_overview (EDA), including handling of missing data, removal of low‑quality fields, type corrections, logical consistency checks, and creation of derived features.

### Steps
- Load pre‑saved datasets
- Apply preprocessing pypline
- Save cleaned datasets

--------------------
#### Imports & Setup & Dataset
-------------------

In [1]:
# ---------------------------------------------------------
# Import moduls
# ---------------------------------------------------------


# import standard modules
import pandas as pd
import numpy as np
from pathlib import Path

import sys
from pathlib import Path

In [2]:
# shut off some annoying warnings
import warnings

warnings.filterwarnings("ignore", message="A value is trying to be set on a copy")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# ---------------------------------------------------------
# Setup style
# ---------------------------------------------------------

# show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# visualisation settings
pd.set_option('display.float_format', '{:,.2f}'.format)

In [12]:
# ---------------------------------------------------------
# Load scripts
# ----------------------------------------------------------

%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# found min directory
PROJECT_ROOT = Path("..").resolve()

# maindirectory sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# import function from script
from my_scripts.preprocessing import preprocess
from my_scripts.eda import overview

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
# ---------------------------------------------------------
# Import dataset
# ---------------------------------------------------------

df = pd.read_pickle("../data/dataset.pkl")

print("EU dataset:", df.shape)

EU dataset: (4362869, 75)


In [6]:
df.columns

Index(['ID_NOTICE_CAN', 'TED_NOTICE_URL', 'YEAR', 'ID_TYPE', 'DT_DISPATCH',
       'XSD_VERSION', 'CANCELLED', 'CORRECTIONS', 'B_MULTIPLE_CAE', 'CAE_NAME',
       'CAE_NATIONALID', 'CAE_ADDRESS', 'CAE_TOWN', 'CAE_POSTAL_CODE',
       'CAE_GPA_ANNEX', 'ISO_COUNTRY_CODE', 'ISO_COUNTRY_CODE_GPA',
       'B_MULTIPLE_COUNTRY', 'ISO_COUNTRY_CODE_ALL', 'CAE_TYPE',
       'EU_INST_CODE', 'MAIN_ACTIVITY', 'B_ON_BEHALF',
       'B_INVOLVES_JOINT_PROCUREMENT', 'B_AWARDED_BY_CENTRAL_BODY',
       'TYPE_OF_CONTRACT', 'TAL_LOCATION_NUTS', 'B_FRA_AGREEMENT',
       'FRA_ESTIMATED', 'B_FRA_CONTRACT', 'B_DYN_PURCH_SYST', 'CPV',
       'MAIN_CPV_CODE_GPA', 'ID_LOT', 'ADDITIONAL_CPVS', 'B_GPA',
       'GPA_COVERAGE', 'LOTS_NUMBER', 'VALUE_EURO', 'VALUE_EURO_FIN_1',
       'VALUE_EURO_FIN_2', 'B_EU_FUNDS', 'TOP_TYPE', 'B_ACCELERATED',
       'OUT_OF_DIRECTIVES', 'CRIT_CODE', 'CRIT_PRICE_WEIGHT', 'CRIT_CRITERIA',
       'CRIT_WEIGHTS', 'B_ELECTRONIC_AUCTION', 'NUMBER_AWARDS', 'ID_AWARD',
       'ID_LOT_AWA

--------------
### Apply preprocessing pipeline
-----------

In [16]:
# apply funktion
df_clean = preprocess(df)


In [17]:
# shape of the cleaned dataset
print(df_clean.shape)

(3518937, 12)


In [18]:
# inspekt of the cleaned dataset
overview(df_clean)

,dtype,total,missing_n,missing_%,uniques_n,uniques
YEAR,int16,3518937,0,0.00,9,"[2008, 2009, 2010, 2011, 2012, 2013, 2014, 201..."
ISO_COUNTRY_CODE,str,3518937,0,0.00,33,"[DE, FR, ES, SE, PL, IT, HU, CY, UK, RO, PT, N..."
CAE_TYPE,str,3518937,0,0.00,10,"[8, 3, 1, 6, R, N, 4, 5A, 5, Z]"
TYPE_OF_CONTRACT,str,3518937,0,0.00,3,"[W, U, S]"
TOP_TYPE,str,3518937,0,0.00,10,"[OPE, NIC, RES, NOC, Unknown, COD, AWP, NOP, N..."
MAIN_CPV_CODE_GPA,str,3518937,0,0.00,68,"[Unknown, 38.0, 75.0, 14.0, 30.0, 794.0, 73.0,..."
VALUE_EURO,float64,2352464,1166473,33.15,611605,"[nan, 5204457.56, 4924202.16, 1400000.0, 88398..."
CRIT_PRICE_WEIGHT,float64,1555658,1963279,55.79,167,"[nan, 100.0, 70.0, 95.0, 85.0, 60.0, 45.0, 50...."
NUMBER_OFFERS,float64,3518937,0,0.00,309,"[2.0, 3.0, 1.0, 6.0, 9.0, 5.0, 45.0, 4.0, 10.0..."
DT_DISPATCH,datetime64[us],3518937,0,0.00,3291,"[2007-12-11 00:00:00, 2007-12-27 00:00:00, 200..."


#### Notes: Summary of Data Cleaning Results

Notes: Summary of Data Cleaning Results
1. Dataset Size After Preprocessing
- Before: 4,039,906 rows × 75 columns
- After: 3,518,937 rows × 12 columns
The reduction from 75 to 12 columns results from removing non‑informative fields, consolidating text columns, and retaining only variables relevant for competition and failure‑risk modelling.
Rows with missing NUMBER_OFFERS were removed because the number of bids is essential for defining the target variable and cannot be reconstructed from any other fields.

2. Columns Removed During Preprocessing
Removed due to >40% missing values (except CRIT_PRICE_WEIGHT, retained for analytical importance)
- Winner information (WIN_*)
- Contracting authority details (CAE_*)
- GPA-related fields
- Secondary financial fields (VALUE_EURO_FIN_*, AWARD_VALUE_EURO_FIN_1)
- Award criteria weights (CRIT_*, except CRIT_PRICE_WEIGHT)
- Additional CPVs
- High-cardinality procedural flags (B_MULTIPLE_, B_FRA_, FRA_ESTIMATED, etc.)
- TED_NOTICE_URL

Removed due to irrelevance for competition modelling
- Identifiers (ID_NOTICE_CAN, ID_AWARD, ID_LOT_AWARDED, CONTRACT_NUMBER)
- Non-award information (INFO_ON_NON_AWARD, INFO_UNPUBLISHED)
- Administrative metadata (MAIN_ACTIVITY, EU_INST_CODE)

Removed due to consolidation into a single NLP field
- TITLE - is used for all NLP feature extraction steps (TF‑IDF, SVD, NMF, SVM).

3. Key Variables Retained Despite Missing Values (missing values imputed using median) - are central to competition modelling
- VALUE_EURO - Missing: 39.85%
  - Importance: baseline contract value; used for log-transformations and value bins.
- CRIT_PRICE_WEIGHT- Missing: 60.85%
  - Importance: captures the price–quality balance of tender evaluation; essential for modelling competitiveness and failure risk.

4. Additional Preprocessing Steps
- All categorical missing values replaced with "Unknown".
- Numeric missing values left as NaN and imputed during modelling.
- Integer columns downcast (int64 → int8/int16) to reduce memory usage.

5. Conversion of Categorical Variables
- All categorical variables were explicitly converted to string to ensure correct feature engineering and avoid dtype-related errors.
Converted fields include:
- CAE_TYPE
- TYPE_OF_CONTRACT
- TOP_TYPE

6. Removal all High‑Cardinality Categorical Features

--------------
### Save cleaned datasets

-----------

In [19]:
df_clean.to_pickle("../data/dataset_clean.pkl")